## №3 **(4 балла)**

В файле water.txt представлено 61 наблюдение.

Каждое наблюдение – город в Англии и Уэльсе.

Города дополнительно поделены на северные и южные.

Для каждого города известны средняя годовая смертность на 100000 населения (по данным 1958–1964)

и концентрация кальция в питьевой воде (в частях на миллион).

Чем выше концентрация кальция, тем жёстче вода.

In [65]:
import pandas as pd
from statsmodels.stats.weightstats import _tconfint_generic
import numpy as np

df = pd.read_csv("water.txt", sep="\t")
df.head()

,location,town,mortality,hardness
0,South,Bat,124,105
1,North,Birkenhea,166,17
2,South,Birmingha,146,5
3,North,Blackbur,180,14
4,North,Blackpoo,160,18


In [66]:
print("Список параметров:", list(df.columns))

Список параметров: ['location', 'town', 'mortality', 'hardness']


1) Используя метод .describe() вычислите описательные статистики для северных и южных городов. Сравните средние значения смертности в северных и южных городах и значения концентрации кальция в питьевой воде.

In [67]:
south_desc = df.loc[df["location"] == "South"].describe()
north_desc = df.loc[df["location"] == "North"].describe()

print("SOUTH describe:\n", south_desc)
print("\nNORTH describe:\n", north_desc)

print("\nMeans (South):")
print(df.loc[df["location"] == "South", ["mortality", "hardness"]].mean(numeric_only=True))

print("\nMeans (North):")
print(df.loc[df["location"] == "North", ["mortality", "hardness"]].mean(numeric_only=True))


SOUTH describe:
         mortality    hardness
count   26.000000   26.000000
mean   137.076923   69.769231
std     14.053962   40.360682
min    109.000000    5.000000
25%    125.250000   40.250000
50%    135.500000   75.500000
75%    148.000000   99.750000
max    162.000000  138.000000

NORTH describe:
         mortality   hardness
count   35.000000  35.000000
mean   162.857143  30.400000
std     13.760863  26.134494
min    137.000000   6.000000
25%    155.000000  12.500000
50%    163.000000  17.000000
75%    171.500000  44.000000
max    198.000000  94.000000

Means (South):
mortality    137.076923
hardness      69.769231
dtype: float64

Means (North):
mortality    162.857143
hardness      30.400000
dtype: float64


**Получается, что на севере выше смертность и ниже hardnees**

## ДЛЯ ЗАДАНИЙ 2 И 3

In [71]:
def mean_ci_t(x, alpha=0.05):
    x = pd.to_numeric(x, errors="coerce").dropna()
    n = x.shape[0]
    mean = x.mean()
    std_mean = x.std(ddof=1) / np.sqrt(n)  # стандартная ошибка среднего
    dof = n - 1
    return _tconfint_generic(mean, std_mean, dof, alpha, "two-sided")

# Избегаем проблем с NaN:
df["location"] = df["location"].astype(str).str.strip().str.lower()
df["mortality"] = pd.to_numeric(df["mortality"], errors="coerce")
df["hardness"] = pd.to_numeric(df["hardness"], errors="coerce")

2) Постройте 95% доверительные интервалы для средней годовой смертности по
всем южным и северным городам. Отличаются ли границы интервалов?

In [72]:
for loc in ["south", "north"]:
    sub = df.loc[df["location"] == loc]
    ci_mort = mean_ci_t(sub["mortality"], alpha=0.05)
    print(f"{loc.capitalize()} 95% CI mortality: {ci_mort}")

South 95% CI mortality: (np.float64(131.40040500261244), np.float64(142.7534411512337))
North 95% CI mortality: (np.float64(158.13012110288878), np.float64(167.58416461139694))


**ОТВЕТ:**

Границы отличаются = интервалы не пересекаются == средние гарантированно достаточно разные

=> средняя смертность на севере сильно выше, чем на юге;

3) Постройте 95% доверительные интервалы для средней концентрации кальция в питьевой воде для южных и северных городов. Отличаются ли границы интервалов?

In [73]:
for loc in ["south", "north"]:
    sub = df.loc[df["location"] == loc]
    ci_hard = mean_ci_t(sub["hardness"], alpha=0.05)
    print(f"{loc.capitalize()} 95% CI hardness: {ci_hard}")

South 95% CI hardness: (np.float64(53.467198692036106), np.float64(86.07126284642544))
North 95% CI hardness: (np.float64(21.42248728572426), np.float64(39.37751271427574))


**ОТВЕТ:**

Границы отличаются = интервалы не пересекаются == средние гарантированно достаточно разные

=> концентрация кальция на юге выше, чем на севере.

**4) Какие можно сделать выводы?**

По непересечению доверительных интервалов можно сделать вывод, что средние разнятся достаточно сильно